# 03 – Baseline Models: ARIMA, Prophet, XGBoost

This notebook trains and evaluates three baseline forecasting models:
- **ARIMA** – classical statistical time-series model
- **Prophet** – Facebook/Meta time-series model with trend/seasonality decomposition
- **XGBoost** – gradient-boosted trees with engineered features

Predictions are saved for use in notebook 05 (ensemble).

In [ ]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import json
from pathlib import Path

from src.utils.data_loader import load_processed_data
from src.utils.metrics import calculate_metrics
from src.visualization.plotter import Plotter
from src.config import RESULTS_DIR

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
plotter = Plotter()

In [ ]:
# ── Load processed data ─────────────────────────────────────────────────────
train = load_processed_data('train')
val   = load_processed_data('val')
test  = load_processed_data('test')

print(f'Train: {len(train)} | Val: {len(val)} | Test: {len(test)}')

target_col = 'Close'
y_test = test[target_col]

## A. ARIMA

In [ ]:
from src.models.arima_model import ARIMAModel

arima = ARIMAModel(order=(5, 1, 0))
arima.fit(train[target_col])

arima_preds = arima.predict(steps=len(test))
arima_preds.index = test.index[:len(arima_preds)]

arima_metrics = calculate_metrics(y_test.values[:len(arima_preds)], arima_preds.values)
print('ARIMA metrics:', arima_metrics)

arima.save()

## B. Prophet

In [ ]:
try:
    from src.models.prophet_model import ProphetModel

    prophet = ProphetModel()
    prophet.fit(pd.concat([train, val]), target_col=target_col)

    prophet_forecast = prophet.predict(periods=len(test))
    prophet_preds = prophet_forecast['yhat'].values[-len(test):]

    prophet_metrics = calculate_metrics(y_test.values, prophet_preds)
    print('Prophet metrics:', prophet_metrics)

    prophet.save()
except Exception as e:
    print(f'Prophet skipped: {e}')
    prophet_preds = None
    prophet_metrics = {}

## C. XGBoost

In [ ]:
from src.models.xgboost_model import XGBoostModel

feature_cols = [c for c in train.columns if c not in [target_col, 'Open', 'High', 'Low', 'Adj Close']]

X_train = train[feature_cols].fillna(0)
y_train_xgb = train[target_col]
X_val   = val[feature_cols].fillna(0)
y_val_xgb = val[target_col]
X_test  = test[feature_cols].fillna(0)

xgb_model = XGBoostModel()
xgb_model.fit(X_train, y_train_xgb, X_val=X_val, y_val=y_val_xgb)

xgb_preds = xgb_model.predict(X_test)
xgb_metrics = calculate_metrics(y_test.values, xgb_preds)
print('XGBoost metrics:', xgb_metrics)

xgb_model.save()

In [ ]:
# Feature importance
importance = xgb_model.get_feature_importance()
plotter.plot_feature_importance(importance, top_n=20, filename='feature_importance.png')
print('Feature importance chart saved.')

## D. Comparison

In [ ]:
all_metrics = {
    'ARIMA':   arima_metrics,
    'XGBoost': xgb_metrics,
}
if prophet_metrics:
    all_metrics['Prophet'] = prophet_metrics

metrics_df = pd.DataFrame(all_metrics).T
print(metrics_df.to_string())
metrics_df.to_csv(RESULTS_DIR / 'baseline_metrics.csv')

In [ ]:
# Predictions comparison plot
predictions = {'ARIMA': arima_preds.values, 'XGBoost': xgb_preds}
if prophet_preds is not None:
    predictions['Prophet'] = prophet_preds

plotter.plot_predictions_comparison(
    y_test.values,
    predictions,
    dates=test.index,
    title='Baseline Models vs Actual (Test Set)',
    filename='baseline_predictions.png'
)
print('Predictions comparison chart saved.')

In [ ]:
# Save predictions for ensemble
preds_df = pd.DataFrame({'actual': y_test.values, 'arima': arima_preds.values, 'xgboost': xgb_preds}, index=test.index)
if prophet_preds is not None:
    preds_df['prophet'] = prophet_preds
preds_df.to_csv(RESULTS_DIR / 'baseline_predictions.csv')
print('Predictions saved to reports/results/baseline_predictions.csv')

## Summary

See `reports/results/baseline_metrics.csv` for a full metrics table.

Continue to **04_model_lstm.ipynb** for the deep learning model.